# UIT-VSFC Framework Comparison

Evaluates the full stacked-ensemble framework (5 base learners + Gemma meta-model)
on **UIT-VSFC** (Vietnamese Students' Feedback Corpus) under two strategies:

| Strategy | Description |
|---|---|
| **Zero-shot** | Use saved VSMEC artifacts as-is; collapse 7-class emotion probs to 3-class sentiment |
| **Light adapt** | Freeze BERT backbones; refit sklearn on VSFC (3-class); recompute weights on VSFC val; continue LoRA on VSFC |

Two framework systems are compared under each strategy:

| Framework | Description |
|---|---|
| **Weighted 5-base ensemble** | Accuracy-normalized weighted average of 5 base model probability distributions |
| **Stacked Gemma meta-model** | QLoRA-fine-tuned Gemma-2-9B-it that reads the structured stack prompt |

**Base learners** (5 total):
- PhoBERT-v2, CafeBERT, ViBERT (7-class emotion BERT classifiers, always frozen)
- TF-IDF + LogisticRegression, TF-IDF + CalibratedLinearSVC

Headline metrics: **weighted F1**, **macro-F1**, and **MCC** on VSFC test set.

---
**Google Colab:** mount Drive first. Set `MODEL_DIRS` and `ENSEMBLE_ARTIFACTS_DIR` in Section 3. GPU recommended.

## 0. Install dependencies

In [ ]:
%pip install -q datasets transformers torch scikit-learn pandas numpy matplotlib seaborn
%pip install -q peft trl bitsandbytes accelerate

## 1. Imports and repo setup

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Repo root ────────────────────────────────────────────────────────────────
REPO_ROOT = os.environ.get('TM_REPO_ROOT', '/content/drive/MyDrive/thesis/topicmodeling')
TM_ROOT = Path(REPO_ROOT) / 'tm_research'

if 'google.colab' in sys.modules:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
else:
    _here = Path.cwd().resolve()
    while _here.name != 'topicmodeling' and _here.parent != _here:
        _here = _here.parent
    REPO_ROOT = str(_here)
    TM_ROOT = _here / 'tm_research'

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from tm_research.eval.sentiment_collapse import (
    EMOTION_TO_POLARITY, POLARITY_CLASSES,
    collapse_batch, probs_to_pred_labels, evaluate_sentiment,
)
from tm_research.eval.vsfc_framework import (
    collect_bert_probs, get_bert_class_names,
    build_sklearn_vsmec, build_sklearn_vsfc,
    weighted_sentiment_zeroshot, weighted_sentiment_adapted,
    compute_vsfc_weights, build_vsfc_meta_jsonl,
    run_gemma_zeroshot, run_gemma_adapted,
    continue_lora_vsfc, VSFC_SYSTEM_PROMPT,
)
from tm_research.eval.vsfc_format_prompt import parse_label_3class
from tm_research.ensemble.utils_io import load_label_map, load_vsfc_splits

print('REPO_ROOT:', REPO_ROOT)
print('POLARITY_CLASSES:', POLARITY_CLASSES)

## 2. Load UIT-VSFC dataset

> **Prerequisite:** Run `VSFC_DataPreprocessing.ipynb` first to generate the cleaned
> CSVs under `tm_research/data/processed/vsfc/`. Those files apply the same Vietnamese
> text cleaning pipeline as VSMEC, which aligns BERT input distribution with training.

In [ ]:
vsfc_train, vsfc_val, vsfc_test, _vsfc_lmap = load_vsfc_splits()

TRAIN_TEXTS  = vsfc_train['text'].tolist()
TRAIN_LABELS = vsfc_train['label'].tolist()
VAL_TEXTS    = vsfc_val['text'].tolist()
VAL_LABELS   = vsfc_val['label'].tolist()
TEST_TEXTS   = vsfc_test['text'].tolist()
TEST_LABELS  = vsfc_test['label'].tolist()

print(f'Train {len(TRAIN_TEXTS):,} | Val {len(VAL_TEXTS):,} | Test {len(TEST_TEXTS):,}')
print('Test label distribution:')
print(vsfc_test['label'].value_counts())

In [ ]:
# Sanity visualisation: VSFC test sentiment distribution
fig, ax = plt.subplots(figsize=(5, 3))
counts = vsfc_test['label'].value_counts()[POLARITY_CLASSES]
ax.bar(POLARITY_CLASSES, counts.values)
ax.set_title('UIT-VSFC Test — sentiment distribution')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 3. Configure paths

Set `MODEL_DIRS` to point at the BERT emotion classifier directories (each containing
`config.json`, model weights, and `label_mappings.json`).

Set `ENSEMBLE_ARTIFACTS_DIR` to the ensemble pipeline output folder.

In [ ]:
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

_DRIVE_BASE = '/content/drive/MyDrive/thesis'

# BERT emotion classifier checkpoints (7-class, trained on UIT-VSMEC)
# Set a path to None to skip that model.
MODEL_DIRS: Dict[str, Optional[str]] = {
    'phobert': os.environ.get('TM_PHOBERT_DIR',  f'{_DRIVE_BASE}/emotion_classifier_phobert'),
    'cafebert': os.environ.get('TM_CAFEBERT_DIR', f'{_DRIVE_BASE}/emotion_classifier_cafebert'),
    'vibert':   os.environ.get('TM_VIBERT_DIR',   f'{_DRIVE_BASE}/emotion_classifier_vibert'),
}

# Ensemble pipeline artifacts (weights.json, lora_adapter/, label_map.json)
ENSEMBLE_ARTIFACTS_DIR = Path(os.environ.get(
    'TM_ENSEMBLE_ARTIFACTS_DIR',
    f'{_DRIVE_BASE}/topicmodeling/tm_research/ensemble/artifacts'
))
LORA_DIR     = ENSEMBLE_ARTIFACTS_DIR / 'lora_adapter'
WEIGHTS_JSON = ENSEMBLE_ARTIFACTS_DIR / 'weights.json'

# Cache: BERT prob arrays, adapted weights, VSFC LoRA
CACHE_DIR    = TM_ROOT / 'eval' / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
VSFC_LORA_DIR = CACHE_DIR / 'vsfc_lora_adapter'
VSFC_META_DIR = CACHE_DIR / 'vsfc_meta_jsonl'

print('=== Artifact availability ===')
for name, d in MODEL_DIRS.items():
    ok = d is not None and Path(d).exists()
    print(f'  {name:10s}: {"OK  " + str(d) if ok else "MISSING (" + str(d) + ")"}')
print(f'  weights.json: {"OK" if WEIGHTS_JSON.exists() else "MISSING"}')
print(f'  lora_adapter: {"OK" if LORA_DIR.exists() else "MISSING"}')

## 4. Collapse sanity check

In [ ]:
# Verify collapse logic with known examples before running inference.
# Labels in VSMEC alphabetical order: Anger, Disgust, Enjoyment, Fear, Other, Sadness, Surprise
DEMO_CLASS_NAMES = ['Anger', 'Disgust', 'Enjoyment', 'Fear', 'Other', 'Sadness', 'Surprise']

p_pos = np.array([0.01, 0.01, 0.90, 0.01, 0.03, 0.02, 0.02], dtype=np.float32)  # Enjoyment → positive
p_neg = np.array([0.40, 0.05, 0.02, 0.30, 0.05, 0.10, 0.08], dtype=np.float32)  # Anger+Fear → negative
p_neu = np.array([0.05, 0.02, 0.05, 0.03, 0.80, 0.03, 0.02], dtype=np.float32)  # Other → neutral

batch = np.vstack([p_pos, p_neg, p_neu])
collapsed = collapse_batch(batch, DEMO_CLASS_NAMES)
preds = probs_to_pred_labels(collapsed)

for i, (c, pred) in enumerate(zip(collapsed, preds)):
    print(f'  case {i+1}: {c.round(3)} → {pred}')

assert preds[0] == 'positive'
assert preds[1] == 'negative'
assert preds[2] == 'neutral'
print('All sanity checks passed!')

## 5. Collect base model probabilities

Shared setup for both strategies: run BERT checkpoints on VSFC test (and later val)
via the cached inference helper. BERT probs are always 7-class.

In [ ]:
# VSMEC label map (7 emotions) — used for zero-shot Gemma prompts and collapse.
vsmec_label_map = load_label_map()
VSMEC_CLASS_NAMES = vsmec_label_map.class_names
print('VSMEC classes:', VSMEC_CLASS_NAMES)

# VSMEC ensemble weights (accuracy-normalized, from notebook 05).
if WEIGHTS_JSON.exists():
    with open(WEIGHTS_JSON) as f:
        _wj = json.load(f)
    VSMEC_WEIGHTS: Dict[str, float] = _wj.get('weights', _wj)
    print('VSMEC weights:', VSMEC_WEIGHTS)
else:
    VSMEC_WEIGHTS = {}
    print('WARNING: weights.json not found — zero-shot ensemble unavailable.')

In [ ]:
# ── BERT test probs (7-class, cached) ────────────────────────────────────────
print('=== Collecting BERT test probs (7-class) ===')
bert_test_probs_7: Dict[str, np.ndarray] = collect_bert_probs(
    MODEL_DIRS, TEST_TEXTS, CACHE_DIR, split_tag='test', device=DEVICE,
)
print(f'Available BERT test probs: {list(bert_test_probs_7.keys())}')

## 6. Strategy 1 — Zero-shot

Use VSMEC-trained artifacts directly on VSFC test. No VSFC training data is used.

- **BERT models**: frozen 7-class emotion classifiers; probs collapsed to 3-class.
- **Sklearn models**: refitted on UIT-VSMEC train (7-class), applied to VSFC.
- **Weights**: loaded from `weights.json` (VSMEC val accuracy).
- **Gemma meta**: VSMEC LoRA adapter; 7-class emotion output → polarity collapse.

In [ ]:
# ── Sklearn on VSMEC train (7-class) ─────────────────────────────────────────
print('=== Fitting sklearn on VSMEC train (7-class) ===')

try:
    from tm_research.ensemble.utils_io import load_splits
    _vsmec_train, _, _, _vsmec_lmap = load_splits()
    _sk_logreg_zs, _sk_svc_zs, _sk_class_names_zs = build_sklearn_vsmec(
        _vsmec_train, _vsmec_lmap.label2id
    )
    sklearn_test_probs_7_zs: Dict[str, np.ndarray] = {
        'logreg': _sk_logreg_zs.predict_proba(TEST_TEXTS).astype(np.float32),
        'svc':    _sk_svc_zs.predict_proba(TEST_TEXTS).astype(np.float32),
    }
    print('  sklearn 7-class test probs computed.')
except Exception as e:
    sklearn_test_probs_7_zs = {}
    print(f'  Could not build VSMEC sklearn models: {e}')

In [ ]:
# ── Zero-shot: Weighted 5-base ensemble ──────────────────────────────────────
RESULTS: list = []

if bert_test_probs_7 and VSMEC_WEIGHTS:
    print('=== [ZS] Weighted 5-base ensemble ===')
    try:
        zs_ens_3 = weighted_sentiment_zeroshot(
            bert_probs_7     = bert_test_probs_7,
            sklearn_probs_7  = sklearn_test_probs_7_zs,
            class_names_7    = VSMEC_CLASS_NAMES,
            weights          = VSMEC_WEIGHTS,
        )
        zs_ens_preds = probs_to_pred_labels(zs_ens_3)
        zs_ens_metrics = evaluate_sentiment(TEST_LABELS, zs_ens_preds)
        print(f"  weighted_F1: {zs_ens_metrics['weighted_f1']:.4f}  macro_F1: {zs_ens_metrics['macro_f1']:.4f}  MCC: {zs_ens_metrics['mcc']:.4f}")
        RESULTS.append({
            'framework': 'Weighted ensemble',
            'strategy': 'zero_shot',
            **zs_ens_metrics,
        })
    except Exception as e:
        print(f'  Error: {e}')
        zs_ens_preds = None
else:
    zs_ens_preds = None
    print('  Skipping (no base probs or weights available).')

In [ ]:
# ── Zero-shot: Stacked Gemma meta-model ──────────────────────────────────────
SKIP_GEMMA_ZS = not LORA_DIR.exists()

if SKIP_GEMMA_ZS:
    print(f'Skipping Gemma ZS (lora_adapter not found at {LORA_DIR}).')
    zs_gemma_preds = None
else:
    print('=== [ZS] Stacked Gemma meta-model ===')
    _lora_meta_path = ENSEMBLE_ARTIFACTS_DIR / 'lora_train_meta.json'
    with open(_lora_meta_path) as f:
        _lora_meta = json.load(f)
    _gemma_base  = _lora_meta.get('base_model', 'google/gemma-2-9b-it')
    _sys_prompt  = _lora_meta['system_prompt']

    # Collect all available 7-class probs for Gemma prompts
    _all_probs_7 = {**bert_test_probs_7, **sklearn_test_probs_7_zs}

    zs_gemma_preds = run_gemma_zeroshot(
        texts            = TEST_TEXTS,
        base_probs_7     = _all_probs_7,
        weights          = VSMEC_WEIGHTS,
        vsmec_label_map  = vsmec_label_map,
        lora_dir         = LORA_DIR,
        system_prompt    = _sys_prompt,
        gemma_base_model = _gemma_base,
        device           = DEVICE,
    )
    _n_fail = sum(1 for p in zs_gemma_preds if p is None)
    print(f'  Parse failures: {_n_fail}/{len(TEST_TEXTS)}')

    _g_true = [t for t, p in zip(TEST_LABELS, zs_gemma_preds) if p is not None]
    _g_pred = [p for p in zs_gemma_preds if p is not None]
    zs_gemma_metrics = evaluate_sentiment(_g_true, _g_pred)
    print(f"  weighted_F1: {zs_gemma_metrics['weighted_f1']:.4f}  macro_F1: {zs_gemma_metrics['macro_f1']:.4f}  MCC: {zs_gemma_metrics['mcc']:.4f}")
    RESULTS.append({
        'framework': 'Stacked Gemma',
        'strategy': 'zero_shot',
        **zs_gemma_metrics,
    })

## 7. Strategy 2 — Light Adaptation

- BERT backbones are **frozen** (only inference, same checkpoints as Strategy 1).
- Sklearn bases are **retrained natively on VSFC train** (3-class: negative/neutral/positive).
- Ensemble weights are **recomputed on VSFC validation** (3-class accuracy per model).
- Weighted ensemble averages in **3-class space** (collapse BERT 7→3, then average with sklearn 3-class).
- Gemma LoRA is **continued** from VSMEC adapter on VSFC JSONL (3-class completions).

### 7a. Fit sklearn on VSFC train (3-class native)

In [ ]:
print('=== [LA] Fitting sklearn on VSFC train (3-class) ===')
_sk_logreg_la, _sk_svc_la = build_sklearn_vsfc(TRAIN_TEXTS, TRAIN_LABELS)

# Test probs (3-class, sklearn output order matches scikit's sorted classes)
# sklearn sorts classes lexicographically: negative=0, neutral=1, positive=2
sklearn_test_probs_3_la: Dict[str, np.ndarray] = {
    'logreg': _sk_logreg_la.predict_proba(TEST_TEXTS).astype(np.float32),
    'svc':    _sk_svc_la.predict_proba(TEST_TEXTS).astype(np.float32),
}

# Val probs (needed for weight recomputation)
sklearn_val_probs_3_la: Dict[str, np.ndarray] = {
    'logreg': _sk_logreg_la.predict_proba(VAL_TEXTS).astype(np.float32),
    'svc':    _sk_svc_la.predict_proba(VAL_TEXTS).astype(np.float32),
}
print('  sklearn VSFC 3-class probs computed.')

### 7b. Recompute ensemble weights on VSFC validation

In [ ]:
# ── BERT val probs (7-class, cached) ─────────────────────────────────────────
print('=== [LA] Collecting BERT val probs (7-class) ===')
bert_val_probs_7: Dict[str, np.ndarray] = collect_bert_probs(
    MODEL_DIRS, VAL_TEXTS, CACHE_DIR, split_tag='val', device=DEVICE,
)

print('=== [LA] Computing VSFC ensemble weights ===')
VSFC_WEIGHTS: Dict[str, float] = compute_vsfc_weights(
    bert_probs_7_val    = bert_val_probs_7,
    sklearn_probs_3_val = sklearn_val_probs_3_la,
    class_names_7       = VSMEC_CLASS_NAMES,
    gold_val            = VAL_LABELS,
)
print('  VSFC weights:', VSFC_WEIGHTS)

# Save for reproducibility
_vsfc_weights_path = CACHE_DIR / 'weights_vsfc.json'
with open(_vsfc_weights_path, 'w') as f:
    json.dump(VSFC_WEIGHTS, f, indent=2)
print(f'  Saved → {_vsfc_weights_path}')

### 7c. Light-adapt weighted ensemble → test

In [ ]:
if bert_test_probs_7:
    print('=== [LA] Weighted 5-base ensemble ===')
    try:
        la_ens_3 = weighted_sentiment_adapted(
            bert_probs_7      = bert_test_probs_7,
            sklearn_probs_3   = sklearn_test_probs_3_la,
            class_names_7     = VSMEC_CLASS_NAMES,
            weights           = VSFC_WEIGHTS,
        )
        la_ens_preds = probs_to_pred_labels(la_ens_3)
        la_ens_metrics = evaluate_sentiment(TEST_LABELS, la_ens_preds)
        print(f"  weighted_F1: {la_ens_metrics['weighted_f1']:.4f}  macro_F1: {la_ens_metrics['macro_f1']:.4f}  MCC: {la_ens_metrics['mcc']:.4f}")
        RESULTS.append({
            'framework': 'Weighted ensemble',
            'strategy': 'light_adapt',
            **la_ens_metrics,
        })
    except Exception as e:
        print(f'  Error: {e}')
        la_ens_preds = None
else:
    la_ens_preds = None
    print('  Skipping (no BERT test probs).')

### 7d. Build VSFC meta JSONL for LoRA continuation

In [ ]:
SKIP_LORA_CONTINUE = not LORA_DIR.exists()

if SKIP_LORA_CONTINUE:
    print(f'Skipping LoRA continuation (lora_adapter not found at {LORA_DIR}).')
else:
    print('=== [LA] Building VSFC meta JSONL ===')

    # BERT train probs (7-class → collapse to 3 inside builder)
    print('  Collecting BERT train probs (this may take a while) …')
    bert_train_probs_7: Dict[str, np.ndarray] = collect_bert_probs(
        MODEL_DIRS, TRAIN_TEXTS, CACHE_DIR, split_tag='train', device=DEVICE,
    )

    # Collapse BERT train + val probs to 3-class for the JSONL
    def _collapse_all(probs_7_dict):
        return {
            k: collapse_batch(v, VSMEC_CLASS_NAMES)
            for k, v in probs_7_dict.items()
        }

    bert_train_probs_3 = _collapse_all(bert_train_probs_7)
    bert_val_probs_3   = _collapse_all(bert_val_probs_7)

    # sklearn train probs (3-class)
    sklearn_train_probs_3_la: Dict[str, np.ndarray] = {
        'logreg': _sk_logreg_la.predict_proba(TRAIN_TEXTS).astype(np.float32),
        'svc':    _sk_svc_la.predict_proba(TRAIN_TEXTS).astype(np.float32),
    }

    VSFC_META_DIR.mkdir(parents=True, exist_ok=True)
    vsfc_train_jsonl = build_vsfc_meta_jsonl(
        texts           = TRAIN_TEXTS,
        bert_probs_3    = bert_train_probs_3,
        sklearn_probs_3 = sklearn_train_probs_3_la,
        weights         = VSFC_WEIGHTS,
        labels          = TRAIN_LABELS,
        jsonl_path      = VSFC_META_DIR / 'train.jsonl',
        system_prompt   = VSFC_SYSTEM_PROMPT,
    )
    vsfc_val_jsonl = build_vsfc_meta_jsonl(
        texts           = VAL_TEXTS,
        bert_probs_3    = bert_val_probs_3,
        sklearn_probs_3 = sklearn_val_probs_3_la,
        weights         = VSFC_WEIGHTS,
        labels          = VAL_LABELS,
        jsonl_path      = VSFC_META_DIR / 'val.jsonl',
        system_prompt   = VSFC_SYSTEM_PROMPT,
    )

### 7e. Continue LoRA on VSFC

Loads the VSMEC LoRA adapter and continues training on VSFC train JSONL with a
lower learning rate (5e-5 vs. 1e-4 original). Set `SKIP_LORA_CONTINUE = True`
to skip this step and load a previously saved adapter from `VSFC_LORA_DIR`.

In [ ]:
if not SKIP_LORA_CONTINUE:
    if VSFC_LORA_DIR.exists():
        print(f'VSFC LoRA adapter already exists at {VSFC_LORA_DIR} — skipping continuation.')
        print('  (Delete the directory and rerun to retrain.)')
    else:
        print('=== [LA] Continuing LoRA on VSFC ===')
        _lora_meta_path = ENSEMBLE_ARTIFACTS_DIR / 'lora_train_meta.json'
        with open(_lora_meta_path) as f:
            _lora_meta_la = json.load(f)
        _gemma_base_la = _lora_meta_la.get('base_model', 'google/gemma-2-9b-it')

        continue_lora_vsfc(
            train_jsonl      = vsfc_train_jsonl,
            val_jsonl        = vsfc_val_jsonl,
            output_dir       = VSFC_LORA_DIR,
            base_model_name  = _gemma_base_la,
            num_epochs       = 2,
            learning_rate    = 5e-5,
            per_device_batch = 4,
            grad_accum       = 4,
        )
        print('  LoRA continuation complete.')

### 7f. Light-adapt Gemma → test

In [ ]:
SKIP_GEMMA_LA = not VSFC_LORA_DIR.exists()

if SKIP_GEMMA_LA:
    print(f'Skipping adapted Gemma (VSFC adapter not found at {VSFC_LORA_DIR}).')
    la_gemma_preds = None
else:
    print('=== [LA] Stacked Gemma meta-model (VSFC-adapted) ===')

    # 3-class probs per model for test
    bert_test_probs_3 = {
        k: collapse_batch(v, VSMEC_CLASS_NAMES)
        for k, v in bert_test_probs_7.items()
    }
    la_test_probs_3 = {**bert_test_probs_3, **sklearn_test_probs_3_la}

    la_gemma_preds = run_gemma_adapted(
        texts         = TEST_TEXTS,
        base_probs_3  = la_test_probs_3,
        weights       = VSFC_WEIGHTS,
        vsfc_lora_dir = VSFC_LORA_DIR,
        system_prompt = VSFC_SYSTEM_PROMPT,
        device        = DEVICE,
    )
    _n_fail = sum(1 for p in la_gemma_preds if p is None)
    print(f'  Parse failures: {_n_fail}/{len(TEST_TEXTS)}')

    _la_g_true = [t for t, p in zip(TEST_LABELS, la_gemma_preds) if p is not None]
    _la_g_pred = [p for p in la_gemma_preds if p is not None]
    la_gemma_metrics = evaluate_sentiment(_la_g_true, _la_g_pred)
    print(f"  weighted_F1: {la_gemma_metrics['weighted_f1']:.4f}  macro_F1: {la_gemma_metrics['macro_f1']:.4f}  MCC: {la_gemma_metrics['mcc']:.4f}")
    RESULTS.append({
        'framework': 'Stacked Gemma',
        'strategy': 'light_adapt',
        **la_gemma_metrics,
    })

## 8. Results

In [ ]:
if not RESULTS:
    print('No results collected. Check that at least one checkpoint is available.')
else:
    results_df = pd.DataFrame(RESULTS)
    display_cols = ['framework', 'strategy', 'weighted_f1', 'macro_f1', 'mcc',
                    'f1_negative', 'f1_neutral', 'f1_positive']
    display_cols = [c for c in display_cols if c in results_df.columns]

    print('=== Framework Comparison on UIT-VSFC Test ===')
    print(results_df[display_cols].to_string(index=False, float_format='{:.4f}'.format))

    _out_csv = CACHE_DIR / 'vsfc_framework_results.csv'
    results_df.to_csv(_out_csv, index=False)
    print(f'\nSaved → {_out_csv}')

In [ ]:
# ── Bar charts: weighted F1, macro-F1, MCC by framework × strategy ───────────
if RESULTS:
    _df = pd.DataFrame(RESULTS)
    _frameworks = _df['framework'].unique().tolist()
    _strategies = ['zero_shot', 'light_adapt']
    _strategy_labels = {'zero_shot': 'Zero-shot', 'light_adapt': 'Light adapt'}
    x = np.arange(len(_frameworks))
    width = 0.35
    colors = ['#4C9BE8', '#E87C4C']

    def _plot_framework_metric(metric_col, ylabel, title):
        fig, ax = plt.subplots(figsize=(8, 5))
        for i, strat in enumerate(_strategies):
            _sub = _df[_df['strategy'] == strat].set_index('framework')
            vals = [_sub.loc[fw, metric_col] if fw in _sub.index else 0.0 for fw in _frameworks]
            bars = ax.bar(x + (i - 0.5) * width, vals, width, label=_strategy_labels[strat],
                          color=colors[i], alpha=0.85)
            for bar, v in zip(bars, vals):
                if v > 0:
                    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.005, f'{v:.3f}',
                            ha='center', va='bottom', fontsize=9)
        ax.set_xticks(x)
        ax.set_xticklabels(_frameworks, fontsize=11)
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.set_ylim(0, 1.0)
        ax.legend()
        for spine in ['top', 'right']:
            ax.spines[spine].set_visible(False)
        plt.tight_layout()
        plt.show()

    _plot_framework_metric('weighted_f1', 'Weighted F1 (3-class sentiment)',
                           'UIT-VSFC: Weighted F1 — Framework × Strategy')
    _plot_framework_metric('macro_f1', 'Macro-F1 (3-class sentiment)',
                           'UIT-VSFC: Macro-F1 — Framework × Strategy')
    _plot_framework_metric('mcc', 'MCC', 'UIT-VSFC: MCC — Framework × Strategy')

In [ ]:
# ── Confusion matrices ────────────────────────────────────────────────────────
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=POLARITY_CLASSES)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=POLARITY_CLASSES)
    fig, ax = plt.subplots(figsize=(4, 3.5))
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

if 'zs_ens_preds' in dir() and zs_ens_preds is not None:
    plot_cm(TEST_LABELS, zs_ens_preds, 'Weighted ensemble — Zero-shot')

if 'zs_gemma_preds' in dir() and zs_gemma_preds is not None:
    _cm_true = [t for t, p in zip(TEST_LABELS, zs_gemma_preds) if p is not None]
    _cm_pred = [p for p in zs_gemma_preds if p is not None]
    plot_cm(_cm_true, _cm_pred, 'Stacked Gemma — Zero-shot')

if 'la_ens_preds' in dir() and la_ens_preds is not None:
    plot_cm(TEST_LABELS, la_ens_preds, 'Weighted ensemble — Light adapt')

if 'la_gemma_preds' in dir() and la_gemma_preds is not None:
    _cm_true2 = [t for t, p in zip(TEST_LABELS, la_gemma_preds) if p is not None]
    _cm_pred2 = [p for p in la_gemma_preds if p is not None]
    plot_cm(_cm_true2, _cm_pred2, 'Stacked Gemma — Light adapt')

## 9. Discussion

### Published VSFC benchmarks (for context)

| Method | F1 (weighted) |
|---|---|
| MaxEnt classifier (original paper) | ~88% |
| PhoBERT+CNN+LSTM | **92.92%** |
| Thin et al. ensemble (soft voting) | 94.03% weighted / 84.51% macro |

*Note: published VSFC numbers typically use weighted F1; this notebook reports weighted F1, macro-F1, and MCC.*

### Design notes

**Zero-shot 7→3 collapse (Strategy 1):** The BERT emotion classifiers were trained on
social-media text (UIT-VSMEC). The `Surprise` emotion is mapped to neutral (NRC-inspired).
Distribution shift between social-media text and student feedback is expected to
depress performance relative to native VSFC models.

**3-class bridge (Strategy 2 weighted ensemble):** Because VSFC-adapted sklearn models produce
3-class probabilities directly, the weighted average is performed in 3-class space:
each BERT model's 7-class output is collapsed to 3-class first, then all five models
are averaged using VSFC-validation-tuned weights. This preserves the 5-model story
while respecting VSFC's label space.

**LoRA continuation (Strategy 2 Gemma):** The VSMEC adapter is continued for 2 epochs
at a lower learning rate (5e-5 vs 1e-4 original) to avoid catastrophic forgetting of
the structured-prompt reading ability. The prompt schema is updated to use 3-class
probability distributions and a sentiment-specific system prompt.